### ML Code
Time to make some models! 


In [ ]:
# First we got to do some imports (I'm just grabbing the ones from 02 for now)
from pathlib import Path
import platform
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.metrics import mean_squared_error, r2_score

# Random seed for reproducibility
np.random.seed(42)

df = pd.read_csv("data/ecowas_df_synthetic_full.csv")

We want to make XGBoost, Random Forest and Neural Networks

In [20]:
from xgboost import XGBRegressor

from sklearn.ensemble import RandomForestRegressor

from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.pipeline import Pipeline



legal_target_vars = {"tradeflow_baci", 'tradeflow_comtrade_o', 'tradeflow_comtrade_d', 'tradeflow_imf_o', 'tradeflow_imf_d', "combined_trade_baci"}
required_columns = {"gdp_o", "gdp_d", "distw_arithmetic"}


## XGBoost implementation

In [ ]:
def xgboost_regression(
    df: pd.DataFrame,
    target_variable: str,
    columns: list,
    year_split: int = 2016,
    set_random_state: int = 42,
    n_estimators: int = 500,
    learning_rate: float = 0.05,
    max_depth: int = 6,
    subsample: float = 0.8,
    colsample_bytree: float = 0.8,
):
    """
    XGBoost regression with identical preprocessing to multilayer_perceptron
    """

    columns = columns.copy()

    # -----------------------------
    # Validation checks
    # -----------------------------
    missing_columns = required_columns - set(columns)
    if missing_columns:
        raise ValueError(
            f"Function columns MUST include all of the following columns from dataframe: {required_columns}."
        )

    if target_variable not in legal_target_vars:
        raise ValueError(
            f"Invalid target variable '{target_variable}'.\n"
            f"Target variable must be one of {legal_target_vars}."
        )

    # -----------------------------
    # Train / test split
    # -----------------------------
    train_df_full = df[df["year"] <= year_split].copy()
    test_df_full = df[df["year"] > year_split].copy()

    all_needed = columns + [target_variable]
    train_df = train_df_full[all_needed].dropna().copy()
    test_df = test_df_full[all_needed].dropna().copy()

    # -----------------------------
    # Log transformations
    # -----------------------------
    train_df[f"{target_variable}_log_trade"] = np.log(train_df[target_variable])
    test_df[f"{target_variable}_log_trade"] = np.log(test_df[target_variable])

    for var in ["gdp_o", "gdp_d", "distw_arithmetic"]:
        train_df[f"log_{var}"] = np.log(train_df[var])
        test_df[f"log_{var}"] = np.log(test_df[var])

    # Replace original columns with logs
    update_list = [
        ("gdp_o", "log_gdp_o"),
        ("gdp_d", "log_gdp_d"),
        ("distw_arithmetic", "log_distw_arithmetic"),
    ]

    for original, log_var in update_list:
        columns.remove(original)
        columns.append(log_var)

    # -----------------------------
    # Final model matrices
    # -----------------------------
    model_columns = columns + [f"{target_variable}_log_trade"]

    train_df_model = train_df[model_columns].dropna()
    test_df_model = test_df[model_columns].dropna()

    X_train = train_df_model[columns]
    y_train = train_df_model[f"{target_variable}_log_trade"]

    X_test = test_df_model[columns]
    y_test = test_df_model[f"{target_variable}_log_trade"]

    # -----------------------------
    # XGBoost model
    # -----------------------------
    model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        random_state=set_random_state,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)

    # -----------------------------
    # Evaluation
    # -----------------------------
    y_pred = model.predict(X_test)

    metrics = {
        "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
        "r2": r2_score(y_test, y_pred),
        "n_train": len(X_train),
        "n_test": len(X_test),
    }

    return model, metrics, y_pred

In [17]:
combined_col = ["gdp_o", "gdp_d", "distw_arithmetic", "contig"] + conflict_vars

xgboost_regression(df, "tradeflow_baci", combined_col)

(XGBRegressor(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=-1, num_parallel_tree=None, ...),
 {'rmse': np.float64(1.6493375432816189),
  'r2': 0.31542393114658573,
  'n_train': 3458,
  'n_test': 910},
 array([10.483826 , 11.064876 , 11.266937 ,  9.697451 , 10.265238 ,
         9.593476 ,  8.637036 ,  9.555764 , 11.548187 , 11.297685 ,
 

## Random Forest implementation

In [22]:
def random_forest_regression(
    df: pd.DataFrame,
    target_variable: str,
    columns: list,
    year_split: int = 2016,
    set_random_state: int = 42,
    n_estimators: int = 500,
    max_depth: int | None = None,
    min_samples_leaf: int = 1,
    min_samples_split: int = 2,
    max_features: str | float = "sqrt",
):
    """
    Random Forest regression with identical preprocessing structure
    used in multilayer_perceptron and xgboost_regression.
    """

    columns = columns.copy()

    # -----------------------------
    # Validation checks
    # -----------------------------
    missing_columns = required_columns - set(columns)
    if missing_columns:
        raise ValueError(
            f"Function columns MUST include all of the following columns from dataframe: {required_columns}."
        )

    if target_variable not in legal_target_vars:
        raise ValueError(
            f"Invalid target variable '{target_variable}'.\n"
            f"Target variable must be one of {legal_target_vars}."
        )

    # -----------------------------
    # Train / test split
    # -----------------------------
    train_df_full = df[df["year"] <= year_split].copy()
    test_df_full = df[df["year"] > year_split].copy()

    all_needed = columns + [target_variable]
    train_df = train_df_full[all_needed].dropna().copy()
    test_df = test_df_full[all_needed].dropna().copy()

    # -----------------------------
    # Log transformations
    # -----------------------------
    train_df[f"{target_variable}_log_trade"] = np.log(train_df[target_variable])
    test_df[f"{target_variable}_log_trade"] = np.log(test_df[target_variable])

    for var in ["gdp_o", "gdp_d", "distw_arithmetic"]:
        train_df[f"log_{var}"] = np.log(train_df[var])
        test_df[f"log_{var}"] = np.log(test_df[var])

    # Replace original columns with logs
    update_list = [
        ("gdp_o", "log_gdp_o"),
        ("gdp_d", "log_gdp_d"),
        ("distw_arithmetic", "log_distw_arithmetic"),
    ]

    for original, log_var in update_list:
        columns.remove(original)
        columns.append(log_var)

    # -----------------------------
    # Final model matrices
    # -----------------------------
    model_columns = columns + [f"{target_variable}_log_trade"]

    train_df_model = train_df[model_columns].dropna()
    test_df_model = test_df[model_columns].dropna()

    X_train = train_df_model[columns]
    y_train = train_df_model[f"{target_variable}_log_trade"]

    X_test = test_df_model[columns]
    y_test = test_df_model[f"{target_variable}_log_trade"]

    # -----------------------------
    # Random Forest model
    # -----------------------------
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        min_samples_split=min_samples_split,
        max_features=max_features,
        random_state=set_random_state,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)

    # -----------------------------
    # Evaluation
    # -----------------------------
    y_pred = model.predict(X_test)

    metrics = {
        "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
        "r2": r2_score(y_test, y_pred),
        "n_train": len(X_train),
        "n_test": len(X_test),
    }

    return model, metrics, y_pred

In [23]:
random_forest_regression(df, "tradeflow_baci", combined_col)

(RandomForestRegressor(max_features='sqrt', n_estimators=500, n_jobs=-1,
                       random_state=42),
 {'rmse': np.float64(1.6241464236012477),
  'r2': 0.336175948071913,
  'n_train': 3458,
  'n_test': 910},
 array([10.83782151, 10.37125389, 10.42857634,  9.72311438,  9.17574849,
         9.11898159,  9.1564099 ,  9.81771277, 10.9350623 , 10.94879473,
         9.39192173,  9.3824803 , 10.57045798, 11.5414199 , 11.01177546,
        11.06714844,  9.77631142,  8.53103263,  8.48961253,  8.53093121,
         9.84393858, 11.59548217, 11.46008858,  9.70597953,  8.76346987,
        11.33134568, 11.38904212, 10.96134024, 10.97402265,  9.89624997,
         8.76651045,  8.74257557,  8.8401984 ,  9.93173024, 11.45413787,
        11.30235534,  9.7720649 ,  8.92606221, 11.21065765, 11.80354414,
        11.4550148 , 11.4203247 , 10.49958882,  8.84611241,  8.80819759,
         8.99295067, 10.66644561, 11.84633138, 11.5709541 , 10.38144449,
         9.38827289, 11.30680788, 11.92301451, 12.

## Neural Network implementation

In [ ]:
def multilayer_perceptron(df: pd.DataFrame, target_variable: str, columns: list, year_split = 2016, hidden_layer=(100,50), set_random_state=42):
    '''
    The main function for creating our regression

    
    Input:

    Output:
        
    '''
    columns = columns.copy()
    train_df_full = df[df["year"] <= year_split].copy()
    test_df_full = df[df["year"] > year_split].copy()
    
    missing_columns = required_columns - set(columns)
    if missing_columns:
        raise ValueError(f"Function columns MUST include all of the following columns from dataframe: {required_columns}.")
    
    # We check for validity in input
    if target_variable not in legal_target_vars:    
        raise ValueError(f"Invalid target variable '{target_variable}'. \n"
                         f"Target variable must be one of {legal_target_vars}.")
    
    


    all_needed = columns + [target_variable]
    train_df = train_df_full[all_needed].copy()
    test_df = test_df_full[all_needed].copy()

    train_df = train_df.dropna()
    test_df = test_df.dropna()


    train_df[f"{target_variable}_log_trade"] = np.log(train_df[target_variable])
    test_df[f"{target_variable}_log_trade"]  = np.log(test_df[target_variable])

    train_df["log_gdp_o"] = np.log(train_df["gdp_o"])
    train_df["log_gdp_d"] = np.log(train_df["gdp_d"])
    train_df["log_distw"]  = np.log(train_df["distw_arithmetic"])

    test_df["log_gdp_o"] = np.log(test_df["gdp_o"])
    test_df["log_gdp_d"] = np.log(test_df["gdp_d"])
    test_df["log_distw"]  = np.log(test_df["distw_arithmetic"])

    update_list_holder = ["gdp_o", "log_gdp_o", "gdp_d", "log_gdp_d", "distw_arithmetic", "log_distw"]
    for i in range(0, len(update_list_holder), 2):
        columns.remove(update_list_holder[i])
        columns.append(update_list_holder[i+1])
    

    model_columns = columns + [f"{target_variable}_log_trade"]

    train_df_model = train_df[model_columns].dropna()
    test_df_model = test_df[model_columns].dropna()

    X_train = train_df_model[columns]
    y_train = train_df_model[f"{target_variable}_log_trade"]

    X_test = test_df_model[columns]
    y_test = test_df_model[f"{target_variable}_log_trade"]


    model = Pipeline(
        steps=[
            ("scaler", StandardScaler()), 
            ("mlp", MLPRegressor(hidden_layer_sizes=hidden_layer, activation="relu",  solver="adam", alpha=0.0001, max_iter = 1000, random_state = set_random_state),
            ),
        ]
    )

    model.fit(X_train, y_train)



    y_pred = model.predict(X_test)

    metrics = {
        "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
        "r2": r2_score(y_test, y_pred),
        "n_train": len(X_train),
        "n_test": len(X_test),
    }

    return model, metrics, y_pred

In [14]:

conflict_vars = (
    [c for c in df.columns if c.startswith("disorder_")] +
    [c for c in df.columns if c.startswith("event_")] +
    [c for c in df.columns if c.startswith("perpetrator_")] +
    [c for c in df.columns if c.startswith("target_")] +
    ["fatalities"]
)


In [15]:
(df[conflict_vars] > 0).mean().sort_values()


perpetrator_Civilians                          0.017857
target_Protesters                              0.229167
target_External/Other forces                   0.255952
event_Explosions/Remote violence               0.276786
perpetrator_External/Other forces              0.291667
disorder_Political violence; Demonstrations    0.303571
target_Rebel group                             0.342262
perpetrator_Rebel group                        0.410714
target_Rioters                                 0.425595
target_Identity militia                        0.434524
target_Political militia                       0.479167
perpetrator_Identity militia                   0.511905
disorder_Strategic developments                0.553571
event_Strategic developments                   0.553571
perpetrator_Rioters                            0.741071
perpetrator_Political militia                  0.744048
event_Riots                                    0.750000
event_Battles                                  0

In [ ]:
combined_col = ["gdp_o", "gdp_d", "distw_arithmetic", "contig"] + conflict_vars

model, metrics, preds = multilayer_perceptron(
    df=df,
    target_variable="tradeflow_baci",
    columns=combined_col,
    hidden_layer=(128, 64)
)


In [11]:
metrics

{'rmse': np.float64(3.9343375686667192),
 'r2': -2.895344421284436,
 'n_train': 3458,
 'n_test': 910}